# Day 48 Revision Summary — Serving a Hand-Rolled CNN Behind an API

- **Convolution, ReLU, and max-pooling implemented from scratch with NumPy** — no deep-learning framework. `conv2d` is a padded sliding-window dot product, `relu` clips negatives to zero, `max_pool2d` keeps the largest value in each window. This is the same CNN forward pass used in Day 41/42, now reused rather than rebuilt.
- **Hand-crafted edge-detection kernels stand in for "learned" filters** — four fixed 3x3 kernels (vertical, horizontal, two diagonals) extract directional gradients; each is convolved, ReLU'd, and pooled, then the pooled maps are flattened and concatenated into one feature vector.
- **Preprocessing must exactly match the training distribution.** The model was trained on sklearn's digits format (8x8, pixel values 0–16), so any uploaded image has to be converted to grayscale, resized to 8x8, and rescaled from the PNG range [0, 255] to [0, 16] — get this wrong and inference silently produces nonsense instead of an error.
- **FastAPI's `lifespan` context manager loads the model once at startup**, not per request — `cnn_digit_model.npz` (weights `W`, bias `b`, and the standardization stats `mu`/`sigma`) is loaded into `model_state` on startup and cleared on shutdown, which is the standard pattern for keeping inference fast.
- **Pydantic response models + per-stage timing turn an endpoint into something debuggable.** `ClassifyResponse` gives the `/classify` route a typed, self-documenting JSON contract, and splitting `preprocess_ms` from `inference_ms` makes it possible to tell whether a slow request is an image-decoding problem or a model problem.

## Today's material (Day 48)

Checked all three sources under the `day48` Drive folder:

- **Classwork** (`classwork/classify_api.py`) — a FastAPI service that reuses Day 41/42's exact conv-feature CNN (`conv2d -> ReLU -> max_pool2d -> flatten -> softmax head`) and adds one new piece: `preprocess_image()`, which turns an arbitrary uploaded image into the 8x8, [0,16]-scale format the model was trained on.
- **Homework** — there is no separate `homework` subfolder for Day 48.
- **Slides** — no slide deck was present in the Day 48 folder, so there was no tacked-on homework slide to check.

With no separate homework source, the day's one assignment is the classwork file itself: get `classify_api.py` running end-to-end as a real, testable inference service.

## Assignment — Run the CNN digit classifier as a FastAPI service (`classwork/classify_api.py`)

**What's being asked:** Stand up `/classify`, an endpoint that accepts an uploaded image, preprocesses it into the exact 8x8 / [0,16] grayscale format the Day 41/42 CNN was trained on, runs it through the hand-rolled conv -> ReLU -> pool -> softmax pipeline, and returns the predicted digit with a confidence score and per-class probabilities — plus a `/health` endpoint to confirm the model loaded.

**Approach:**
1. Make sure `cnn_digit_model.npz` exists (trained earlier via Day 41/42's `train_cnn_classifier.py`) and contains the four arrays the service expects: `W`, `b`, `mu`, `sigma`.
2. Install the service's dependencies — `fastapi`, `uvicorn`, `pillow`, `numpy`, `pydantic` — since none of these are needed by the pure-NumPy training script.
3. Confirm `preprocess_image()` does the three things that matter: convert to grayscale (`"L"` mode) to match the training data, resize to 8x8 with `Image.LANCZOS`, and rescale `[0,255] -> [0,16]` since that's the sklearn digits range the model was fit on.
4. Trace `conv_features()`: each of the four kernels (`vertical`, `horizontal`, `diag_main`, `diag_anti`) goes through `conv2d` (padding=1) -> `relu` -> `max_pool2d`, and the four pooled maps get flattened and concatenated into a single feature vector.
5. In the `/classify` handler, check the request really is an image (`content_type` starts with `image/`), read the bytes, and time preprocessing and inference separately so the two costs don't get conflated.
6. Standardize the extracted features with the stored `mu`/`sigma` before the linear layer, then apply `softmax(features @ W + b)` to get a probability over the 10 digit classes, and return the arg-max digit plus its confidence and the full probability dict via the `ClassifyResponse` model.
7. Start the service locally with `uvicorn classify_api:app --port 8000`, hit `/health` to confirm `model_loaded` is true, then POST a real digit image to `/classify` (e.g. through the auto-generated Swagger UI at `/docs`) and sanity-check that the predicted digit and confidence look reasonable.